In [1]:
# Importando as bibliotecas

import pandas as pd 
import numpy as np
import os
from datetime import datetime, date
from google.cloud import bigquery
from dotenv import load_dotenv
import pyarrow


# Carregando variáveis de ambiente
load_dotenv()

True

In [2]:
df_2021_01 = pd.read_csv("/home/danielpedro/Engenharia de Dados/combustivel_preco2015_2025/dataset/ca-2021-01.csv", sep=";",low_memory=False)

In [3]:
df_2021_02 = pd.read_csv(
    "/home/danielpedro/Engenharia de Dados/combustivel_preco2015_2025/dataset/ca-2021-02.csv",
    sep=";",
    encoding="latin1",
    low_memory=False
)

In [4]:
# Concatenando os dataframes de 2021_01 a 2021_02
df_2021 = pd.concat((df_2021_01, df_2021_02))

In [5]:
df_2021.info()

<class 'pandas.core.frame.DataFrame'>
Index: 807537 entries, 0 to 472855
Data columns (total 16 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Regiao - Sigla     807537 non-null  object 
 1   Estado - Sigla     807537 non-null  object 
 2   Municipio          807537 non-null  object 
 3   Revenda            807537 non-null  object 
 4   CNPJ da Revenda    807537 non-null  object 
 5   Nome da Rua        807537 non-null  object 
 6   Numero Rua         807222 non-null  object 
 7   Complemento        165365 non-null  object 
 8   Bairro             805808 non-null  object 
 9   Cep                807537 non-null  object 
 10  Produto            807537 non-null  object 
 11  Data da Coleta     807537 non-null  object 
 12  Valor de Venda     807537 non-null  object 
 13  Valor de Compra    0 non-null       float64
 14  Unidade de Medida  807537 non-null  object 
 15  Bandeira           807537 non-null  object 
dtypes: floa

In [6]:
# Filtro aplicado na extração: apenas registros do estado de Pernambuco (UF == "PE")
# Decisão: reduzir volume de armazenamento e escopo do projeto
df_2021_pe = df_2021[df_2021["Estado - Sigla"] == "PE"]

In [7]:
# Ajustndo o tipo de dado da coluna "Data da Coleta" para datetime
df_2021_pe["Data da Coleta"] = pd.to_datetime(df_2021_pe["Data da Coleta"], errors="coerce")


/tmp/ipykernel_1729/3517031048.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2021_pe["Data da Coleta"] = pd.to_datetime(df_2021_pe["Data da Coleta"], errors="coerce")


In [8]:
# Ajustando o tipo de dado da coluna "Valor de Venda" e "Valor de Compra" para numérico
df_2021_pe["Valor de Venda"] = pd.to_numeric(df_2021_pe["Valor de Venda"], errors="coerce")
df_2021_pe["Valor de Compra"] = pd.to_numeric(df_2021_pe["Valor de Compra"], errors="coerce")


/tmp/ipykernel_1729/811502952.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2021_pe["Valor de Venda"] = pd.to_numeric(df_2021_pe["Valor de Venda"], errors="coerce")
/tmp/ipykernel_1729/811502952.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2021_pe["Valor de Compra"] = pd.to_numeric(df_2021_pe["Valor de Compra"], errors="coerce")


In [9]:
df_2021_pe.head()

,Regiao - Sigla,Estado - Sigla,Municipio,Revenda,CNPJ da Revenda,Nome da Rua,Numero Rua,Complemento,Bairro,Cep,Produto,Data da Coleta,Valor de Venda,Valor de Compra,Unidade de Medida,Bandeira
1658,NE,PE,JABOATAO DOS GUARARAPES,GUARUJA COMERCIO E DISTRIBUICAO DE PETROLEO LTDA,01.457.180/0001-06,ESTRADA DA BATALHA,371,NaN,PRAZERES,54315-010,DIESEL S10,2021-05-01,NaN,NaN,R$ / litro,PETROBRAS DISTRIBUIDORA S.A.
1659,NE,PE,JABOATAO DOS GUARARAPES,GUARUJA COMERCIO E DISTRIBUICAO DE PETROLEO LTDA,01.457.180/0001-06,ESTRADA DA BATALHA,371,NaN,PRAZERES,54315-010,GNV,2021-05-01,NaN,NaN,R$ / m³,PETROBRAS DISTRIBUIDORA S.A.
1660,NE,PE,JABOATAO DOS GUARARAPES,GUARUJA COMERCIO E DISTRIBUICAO DE PETROLEO LTDA,01.457.180/0001-06,ESTRADA DA BATALHA,371,NaN,PRAZERES,54315-010,ETANOL,2021-05-01,NaN,NaN,R$ / litro,PETROBRAS DISTRIBUIDORA S.A.
1661,NE,PE,JABOATAO DOS GUARARAPES,GUARUJA COMERCIO E DISTRIBUICAO DE PETROLEO LTDA,01.457.180/0001-06,ESTRADA DA BATALHA,371,NaN,PRAZERES,54315-010,GASOLINA,2021-05-01,NaN,NaN,R$ / litro,PETROBRAS DISTRIBUIDORA S.A.
1662,NE,PE,JABOATAO DOS GUARARAPES,C.E.S BELTRAO COMERCIO DE COMBUSTIVEIS LTDA,06.187.301/0001-52,RODOVIA PE 07,S/N,KM 21 625,VARGEM FRIA,54125-130,ETANOL,2021-03-01,NaN,NaN,R$ / litro,BRANCA


In [10]:
# Configurando credenciais

# Caminho do arquivo de credenciais GCP
credencial = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

# Identificador do projeto no GCP
project_id = os.getenv("PROJECT_ID")

# Tabela Bronze no BigQuery
table_id = os.getenv("BRONZE_2021")

# Inicialização do cliente BigQuery

# Define a variavel de ambiente
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = credencial

# Cria o cliente Bigquery ao projeto configurado
client = bigquery.Client(project=project_id)

In [11]:
# Criação e carga da tabela BRONZE

# Configuração do job de carga no BigQuery
# WRITE_TRUNCATE garante que a tabela Bronze seja sempre sobrescrita,
# mantendo apenas o retrato mais recente dos dados de origem
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

# Envia o DataFrame para o BigQuery,
# criando a tabela automaticamente caso ainda não exista
job = client.load_table_from_dataframe(
    df_2021_pe,
    table_id,
    job_config=job_config
)

# Aguarda a finalização do job para garantir consistência da carga
job.result()

# Confirmação visual de sucesso da operação
print("✅ Tabela Bronze criada e dados carregados com sucesso")

/home/danielpedro/anaconda3/envs/combustivel_projeto/lib/python3.11/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


✅ Tabela Bronze criada e dados carregados com sucesso
